# Deep Learning: From Neurons to Transformers
## A Foundation for the Attention Is All You Need Workshop

Your Name

---

## Getting Started

- Replace **Your Name** above with your full name
- Automatic 0 if you include your student ID or any other identifying number
- Rename this file `Your_Name.ipynb` before submitting
- **Enable GPU:** Runtime → Change runtime type → T4 GPU (recommended for Parts 2–5)
- When finished, share your Colab link with **Anyone with the link can view** privileges
- Paste the shared link into the Canvas submission

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code to the explanation.
- **Go back and read the code cell** — Returns you to the top of the code section.

**Tips:** Press **H** to jump between headings, **K** or **Tab** to move between links, **D** to jump between landmark regions.

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain how tensors and autograd make neural network training automatic
2. Design hidden layer architectures using principled rules of thumb
3. Build and train an ANN, CNN, RNN, LSTM, and Seq2Seq model in PyTorch
4. Explain what each architecture solves that the previous one could not
5. Implement scaled dot-product attention from scratch
6. Describe the key contributions of the 2017 Transformer paper
7. Speak confidently about neural networks in a technical interview or workshop

---

## The Architecture Progression

Each architecture in this notebook was invented to solve a problem the previous one could not:

| Architecture | Invented | Solves | Still Limited By |
| :--- | :--- | :--- | :--- |
| **ANN** | 1940s–80s | Universal function approximation on tabular data | No spatial or sequential awareness |
| **CNN** | 1989 | Spatial patterns in images | No memory across time |
| **RNN** | 1986 | Sequential data, variable-length inputs | Vanishing gradients over long sequences |
| **LSTM** | 1997 | Long-range dependencies via gated memory | Sequential processing — cannot parallelize |
| **Seq2Seq** | 2014 | Mapping one sequence to another | Fixed-size bottleneck loses long-sequence context |
| **Attention** | 2015 | Dynamic focus on any part of input | Still built on recurrence |
| **Transformer** | 2017 | Full parallelism, unlimited context — attention only | Quadratic memory cost with sequence length |

---

# Part 0: PyTorch Fundamentals

Before building any neural network, you need three PyTorch concepts: **tensors**, **autograd**, and **the training loop**. Everything in Parts 1 through 7 is a variation on this foundation.

---

## Tensors

A tensor is the fundamental data structure in PyTorch — a generalization of arrays to arbitrary dimensions:

- A **scalar** is a 0-dimensional tensor: a single number
- A **vector** is a 1-dimensional tensor: a list of numbers
- A **matrix** is a 2-dimensional tensor: rows and columns
- A **3D tensor** can represent a batch of images, a sequence of word embeddings, or a video frame

The key difference from a numpy array: PyTorch tensors can live on a GPU and can track the operations performed on them for automatic differentiation.

## Autograd

**Autograd** is PyTorch's automatic differentiation engine. When you compute a loss and call `loss.backward()`, PyTorch automatically computes the gradient of the loss with respect to every parameter in the network. You never write a derivative by hand.

This works through the **computation graph**: PyTorch records every operation on tensors that have `requires_grad=True`. When you call `.backward()`, it walks the graph backwards using the chain rule.

## The Training Loop

Every PyTorch training loop follows the same five steps, repeated for each batch:

```
1. optimizer.zero_grad()      # Clear gradients from the previous step
2. output = model(X_batch)    # Forward pass: compute predictions
3. loss = criterion(output, y_batch)  # Compute loss
4. loss.backward()            # Backward pass: compute gradients
5. optimizer.step()           # Update weights using gradients
```

Step 1 is easy to forget but critical — PyTorch accumulates gradients by default, so you must clear them at the start of each batch.

---

<a name="read-code-1"></a>

**Setup Cell — Install and Import Everything**

This cell installs `tqdm` for progress bars and imports all libraries used throughout the notebook: PyTorch core (`torch`, `torch.nn`, `torch.optim`), dataset utilities (`torchvision`, `torchtext`), and standard scientific Python tools. Run this first. If you are on CPU the notebook will still work — training will just be slower for the CNN and RNN sections.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm', '-q'])

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

import torchvision
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import math, random, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

<a name="skip-code-1"></a>

**Expected output:** PyTorch version and device (cuda or cpu). If you see `cpu` and want faster training for CNN/RNN sections, go to Runtime → Change runtime type → T4 GPU.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-2"></a>

**Cell 2 — Tensors: Dimensions, Shapes, and Operations**

This cell demonstrates tensor creation and the most important tensor operations you will encounter: shape inspection, matrix multiplication, and the difference between element-wise and batch operations. The batch dimension (dimension 0) is the one that changes most often — a batch of 32 images shaped `[32, 1, 28, 28]` becomes logits shaped `[32, 10]` after passing through a network.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Scalar, vector, matrix, 3D tensor
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.randn(4, 3)     # 4 rows, 3 columns
tensor3d = torch.randn(2, 4, 3) # batch of 2, each 4x3

print('=== Shapes ===')
print(f'Scalar:   {scalar.shape} — a single number')
print(f'Vector:   {vector.shape} — 3 numbers in a row')
print(f'Matrix:   {matrix.shape} — 4 rows, 3 columns')
print(f'3D:       {tensor3d.shape} — 2 matrices of 4x3')

# Matrix multiplication: [4,3] x [3,2] = [4,2]
W = torch.randn(3, 2)
result = matrix @ W
print(f'\nmatrix @ W : {matrix.shape} x {W.shape} = {result.shape}')

# This is exactly what a Linear layer does (without the bias)
linear = nn.Linear(3, 2, bias=False)
linear.weight = nn.Parameter(W.T)  # nn.Linear stores weights transposed
out = linear(matrix)
print(f'nn.Linear output: {out.shape}  (same as matrix @ W)')

# Batch dimension: same layer, batch of 32
batch = torch.randn(32, 3)
print(f'\nBatch input:  {batch.shape}')
print(f'Batch output: {linear(batch).shape} — 32 predictions, each length 2')

<a name="skip-code-2"></a>

**Expected output:** Shape information and a demonstration that `nn.Linear` is matrix multiplication. The key insight: **the batch dimension is always first in PyTorch** (N, ...). A `nn.Linear(3, 2)` layer takes ANY number of samples, each with 3 features, and produces 2 outputs per sample — regardless of batch size.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-3"></a>

**Cell 3 — Autograd: Gradients Without Writing Derivatives**

This cell demonstrates how PyTorch tracks operations to compute gradients automatically. We compute a simple loss and call `.backward()` — PyTorch walks the computation graph backwards and fills in `.grad` for every tensor that has `requires_grad=True`. This is exactly what happens inside your network during training, applied to millions of parameters.

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# A simple weight with gradient tracking
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)

# Forward pass: compute a prediction and a loss
x      = torch.tensor(3.0)   # input (no gradient needed)
y_true = torch.tensor(10.0)  # target

y_pred = w * x + b           # linear prediction: 2*3 + 0.5 = 6.5
loss   = (y_pred - y_true)**2  # squared error: (6.5 - 10)^2 = 12.25

print(f'Prediction: {y_pred.item():.2f}')
print(f'Loss:       {loss.item():.4f}')

# Backward pass: compute gradients
loss.backward()

print(f'\nd(loss)/d(w) = {w.grad.item():.4f}')
print(f'd(loss)/d(b) = {b.grad.item():.4f}')

# Verify manually:
# d(loss)/dw = 2 * (y_pred - y_true) * x = 2 * (6.5 - 10) * 3 = -21
# d(loss)/db = 2 * (y_pred - y_true) * 1 = 2 * (6.5 - 10) * 1 = -7
print(f'\nManual check: d(loss)/dw = 2*(6.5-10)*3 = {2*(6.5-10)*3:.1f}')
print(f'Manual check: d(loss)/db = 2*(6.5-10)*1 = {2*(6.5-10)*1:.1f}')
print('Autograd matches manually computed gradients.')

<a name="skip-code-3"></a>

**Expected output:** Prediction of 6.5, loss of 12.25, and gradients that match the manual calculation. The negative gradients tell us: decrease w and b to reduce the loss — that is exactly what the optimizer does during training. In a real network, `loss.backward()` does this for every one of the millions of parameters simultaneously.

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-4"></a>

**Cell 4 — The Universal Training Loop**

This cell shows the complete training loop as a reusable function. Every model in this notebook uses a variation of this exact pattern. Study it here so that when you see it again in Parts 1–7, you can focus on the architecture rather than the mechanics.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs=10, device=DEVICE, desc='Training'):
    """
    Universal PyTorch training loop.
    Returns a list of average training loss per epoch.
    """
    model.to(device)
    model.train()
    history = []

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        n_batches  = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # ── The Five Steps ──────────────────────────────────────
            optimizer.zero_grad()           # 1. Clear old gradients
            output = model(X_batch)         # 2. Forward pass
            loss   = criterion(output, y_batch)  # 3. Compute loss
            loss.backward()                 # 4. Backward pass
            optimizer.step()                # 5. Update weights
            # ────────────────────────────────────────────────────────

            epoch_loss += loss.item()
            n_batches  += 1

        avg_loss = epoch_loss / n_batches
        history.append(avg_loss)
        if epoch % max(1, epochs // 5) == 0:
            print(f'  [{desc}] Epoch {epoch:3d}/{epochs}  Loss: {avg_loss:.4f}')

    return history

def evaluate_model(model, loader, device=DEVICE):
    """Compute accuracy on any DataLoader."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            preds   = model(X_batch).argmax(dim=1).cpu()
            correct += (preds == y_batch).sum().item()
            total   += y_batch.size(0)
    return correct / total

def plot_loss(history, title='Training Loss'):
    plt.figure(figsize=(9, 4))
    plt.plot(history, color='steelblue', linewidth=2)
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(title); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print('Training utilities defined.')

<a name="skip-code-4"></a>

**Expected output:** `Training utilities defined.`

The `train_model` and `evaluate_model` functions will be reused in every part. Notice that `model.train()` and `model.eval()` are critical — they toggle behaviors like Dropout (active during training, off during evaluation) and BatchNorm (uses batch statistics during training, running statistics during evaluation). Forgetting `model.eval()` during evaluation is a common bug.

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 1: Artificial Neural Networks and Hidden Layers

An **Artificial Neural Network (ANN)** is a directed graph of artificial neurons organized into layers. Information flows from the input layer, through one or more hidden layers, to the output layer. Each connection has a learnable weight, and each neuron applies an activation function to introduce the non-linearity that makes deep networks powerful.

## The Neuron

A single neuron computes:

$$y = f\left(\sum_{i} w_i x_i + b\right)$$

Where:
- $x_i$ are the inputs
- $w_i$ are the learnable weights
- $b$ is a learnable bias
- $f$ is the activation function

A layer of neurons is just this computation done in parallel — it is a matrix multiplication (`X @ W.T + b`) followed by an activation function.

## Activation Functions

| Function | Formula | Use Case | Why |
| :--- | :--- | :--- | :--- |
| **ReLU** | $\max(0, x)$ | Hidden layers | Fast, no vanishing gradient for positive inputs |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | Binary output | Maps to (0, 1) — probability |
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | RNN hidden states | Zero-centered, (-1, 1) |
| **Softmax** | $\frac{e^{x_i}}{\sum e^{x_j}}$ | Multi-class output | Produces probability distribution |
| **GELU** | Gaussian-smoothed ReLU | Transformers | Smoother than ReLU |

**Default choice for hidden layers:** ReLU. It is computationally simple, avoids the vanishing gradient problem for positive activations, and works well across nearly all architectures.

---

# The Hidden Layer Design Guide

This is the section most textbooks skip. Students frequently ask: *'How many hidden layers? How many neurons per layer?'* There is no single correct answer, but there are principled rules of thumb.

## What Hidden Layers Actually Do

Each hidden layer learns a **progressively more abstract representation** of the input.

Think of it as building a feature hierarchy:

```
Input Layer       → Raw feature values (alcohol: 13.2, proline: 1048)
Hidden Layer 1    → Low-level combinations (high-alcohol AND high-proline)
Hidden Layer 2    → Mid-level patterns (profile consistent with Cultivar A)
Hidden Layer 3    → Abstract decisions (confidence score per class)
Output Layer      → Final probabilities [0.87, 0.09, 0.04]
```

This is the **representation learning** principle: you are not hand-engineering features — you are letting the network discover them. But the network can only discover features that fit inside the hidden layer's capacity.

## The Funnel (Pyramid) Architecture

The most common pattern for classification networks:

```
  Input       ████████████████  (13 features)
  Hidden 1    ████████████      (64 neurons)
  Hidden 2    ██████            (32 neurons)
  Hidden 3    ███               (16 neurons)
  Output      ▌▌▌               (3 classes)
```

Each layer compresses the representation into a smaller, more abstract form. The decreasing sizes force the network to retain only the most useful information.

## Sizing Rules of Thumb

| Rule | Guideline | Example (13 input features, 3 output classes) |
| :--- | :--- | :--- |
| **First hidden layer** | 2× to 4× the input size | 26–52 neurons → use 32 or 64 |
| **Each subsequent layer** | Halve the previous layer | 64 → 32 → 16 |
| **Minimum hidden layer** | Never smaller than the output | At least 3 neurons |
| **Powers of 2** | 16, 32, 64, 128, 256... | GPU memory aligns better |
| **Geometric mean rule** | $\sqrt{n_{\text{input}} \times n_{\text{output}}}$ | $\sqrt{13 \times 3} \approx 6$ (minimum) |

## Width vs Depth

| Approach | Effect | When to Use |
| :--- | :--- | :--- |
| **Wider layers** (more neurons) | More capacity per layer; captures more features simultaneously | When features interact in complex ways at the same level |
| **Deeper network** (more layers) | More abstraction levels; better feature hierarchy | When the problem has hierarchical structure (images, language) |

**Practical rule:** Start with 2–3 hidden layers. Add depth before width — each layer adds a level of abstraction, while width only adds capacity at an existing level. Beyond 4–5 layers on simple data, you typically need skip connections (ResNets) to prevent vanishing gradients.

## Regularization Inside the Network

**Dropout:** Randomly zeroes a fraction of neurons during each training step. Forces the network to learn redundant representations — no single neuron becomes critical. Place after hidden layer activations, before the next layer.

**Batch Normalization:** Normalizes the outputs of a layer across the batch before passing to the next layer. Stabilizes training, allows higher learning rates, acts as mild regularization. Place between the linear layer and the activation function.

```python
# Standard order:
nn.Linear(64, 32),
nn.BatchNorm1d(32),    # Normalize
nn.ReLU(),             # Activate
nn.Dropout(0.3),       # Regularize
nn.Linear(32, 16),
```

## Common Mistakes

| Mistake | What Happens | Fix |
| :--- | :--- | :--- |
| Hidden layer smaller than output | Network bottleneck loses information before prediction | Always keep hidden ≥ output |
| Too few neurons | Underfitting — model too simple to learn the pattern | Double the layer sizes |
| Too many neurons on simple data | Overfitting — memorizes training noise | Add Dropout; reduce size |
| No activation function | Linear layers compose to a single linear layer — depth adds nothing | Always activate between layers |
| Activation after output layer | Distorts probability interpretation | Let the loss function handle the output |

**On that last point:** use `nn.CrossEntropyLoss()` for multi-class classification — it expects raw logits (no Softmax applied) and applies log-softmax internally. Applying Softmax before CrossEntropyLoss is a common bug that causes numerical instability.

---

<a name="read-code-5"></a>

**Cell 5 — Prepare the Wine Dataset as PyTorch Tensors**

This cell loads the 13-feature Wine dataset, scales features to zero mean and unit variance (critical for gradient-based optimization — features on very different scales cause uneven gradients), and wraps everything in PyTorch TensorDatasets and DataLoaders. A batch size of 32 is a good default for small tabular datasets.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Load and prepare data
wine  = load_wine()
X, y  = wine.data.astype(np.float32), wine.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_tr).astype(np.float32)
X_te   = scaler.transform(X_te).astype(np.float32)

# Convert to tensors
X_tr_t = torch.tensor(X_tr);  y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te);  y_te_t = torch.tensor(y_te, dtype=torch.long)

train_ds = TensorDataset(X_tr_t, y_tr_t)
test_ds  = TensorDataset(X_te_t, y_te_t)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=32)

print(f'Input features (n_input):  {X_tr_t.shape[1]}')
print(f'Output classes (n_output): {len(np.unique(y))}')
print(f'Training samples:          {len(X_tr_t)}')
print(f'Test samples:              {len(X_te_t)}')
print(f'Batch shape:               {next(iter(train_dl))[0].shape}')

<a name="skip-code-5"></a>

**Expected output:** 13 input features, 3 output classes, 142 training and 36 test samples. Scaling is essential — without it, the alcohol feature (values ~12–14) and the proline feature (values ~200–1600) would produce gradients of wildly different magnitudes, making the optimizer's job very difficult.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

## Three ANN Architectures to Compare

We will build three versions of an ANN on the same wine data:

1. **Too Narrow** — hidden layers too small to capture the pattern
2. **Well-Designed** — funnel architecture following the rules above
3. **Too Wide** — excess capacity that may overfit

This lets you see the bias-variance tradeoff as a concrete architecture choice.

<a name="read-code-6"></a>

**Cell 6 — Define Three ANN Architectures**

This cell defines a reusable `build_ann()` factory function and creates three networks. The funnel architecture uses 64→32→16 neurons following the rules of thumb: first hidden = ~4× inputs, each layer halves, never below the output size. Dropout(0.3) and BatchNorm are included to show proper placement.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
def build_ann(layer_sizes, dropout=0.0, batch_norm=False):
    """
    Build a fully connected ANN.
    layer_sizes: list of sizes [input, hidden1, ..., output]
    """
    layers = []
    for i in range(len(layer_sizes) - 1):
        layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
        if i < len(layer_sizes) - 2:  # Not the output layer
            if batch_norm:
                layers.append(nn.BatchNorm1d(layer_sizes[i+1]))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)

n_input, n_output = 13, 3

# Architecture 1: Too Narrow
ann_narrow = build_ann([n_input, 4, 4, n_output])
print('Too Narrow:     ', [n_input, 4, 4, n_output])

# Architecture 2: Well-Designed Funnel
ann_funnel = build_ann([n_input, 64, 32, 16, n_output], dropout=0.3, batch_norm=True)
print('Funnel (good):  ', [n_input, 64, 32, 16, n_output])

# Architecture 3: Too Wide
ann_wide = build_ann([n_input, 512, 512, 512, n_output], dropout=0.5)
print('Too Wide:       ', [n_input, 512, 512, 512, n_output])

# Parameter counts
for name, model in [('Narrow', ann_narrow), ('Funnel', ann_funnel), ('Wide', ann_wide)]:
    params = sum(p.numel() for p in model.parameters())
    print(f'  {name:10s}: {params:,} trainable parameters')

<a name="skip-code-6"></a>

**Expected output:** Three architectures with very different parameter counts. The narrow network has hundreds of parameters, the funnel thousands, the wide network hundreds of thousands. On a dataset of 142 training samples, the wide network has far more parameters than samples — a classic overfitting risk.

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-7"></a>

**Cell 7 — Train All Three ANNs and Compare**

This cell trains all three networks for 100 epochs and compares their test accuracy. The funnel network with BatchNorm and Dropout should achieve the best test accuracy despite having fewer parameters than the wide network. The narrow network demonstrates underfitting — too constrained to learn the full pattern.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
criterion = nn.CrossEntropyLoss()
results   = {}

for name, model in [('Narrow', ann_narrow), ('Funnel', ann_funnel), ('Wide', ann_wide)]:
    opt     = optim.Adam(model.parameters(), lr=1e-3)
    history = train_model(model, train_dl, criterion, opt, epochs=100, desc=name)
    train_acc = evaluate_model(model, train_dl)
    test_acc  = evaluate_model(model, test_dl)
    results[name] = {'history': history, 'train': train_acc, 'test': test_acc}
    print(f'  {name}: Train {train_acc:.3f}  Test {test_acc:.3f}  Gap {train_acc-test_acc:+.3f}\n')

# Plot learning curves
fig, ax = plt.subplots(figsize=(11, 5))
colors = {'Narrow': '#e74c3c', 'Funnel': '#27ae60', 'Wide': '#2980b9'}
for name, res in results.items():
    ax.plot(res['history'], label=name, color=colors[name], linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss')
ax.set_title('ANN Learning Curves — Three Architectures', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('ann_learning_curves.png', dpi=150, bbox_inches='tight'); plt.show()

<a name="skip-code-7"></a>

**Expected output:** Training and test accuracy for all three models, and learning curves.

The funnel network typically achieves 95–100% test accuracy. The narrow network will underfit (lower train AND test accuracy). The wide network may achieve near-perfect train accuracy but lower test accuracy — the classic overfitting signature. Notice how Dropout and BatchNorm help the funnel generalize better than the wide network despite the wide network having far more capacity.

**Accessibility note:** Add a Markdown cell describing the learning curves — which network converges fastest? Which shows the smoothest loss curve?

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-8"></a>

**Cell 8 — Visualize the Funnel Architecture**

This cell draws a schematic of the funnel ANN showing the width of each layer. The visual makes the pyramid pattern concrete and gives you a diagram you can reference when explaining hidden layer design.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 10); ax.set_ylim(-0.5, 5.5); ax.axis('off')

layer_info = [
    (1.0, 13,  'Input\n13 features',    '#3498db'),
    (3.0, 64,  'Hidden 1\n64 neurons',   '#8e44ad'),
    (5.0, 32,  'Hidden 2\n32 neurons',   '#8e44ad'),
    (7.0, 16,  'Hidden 3\n16 neurons',   '#8e44ad'),
    (9.0,  3,  'Output\n3 classes',      '#27ae60'),
]

max_n = 64
for x, n, label, color in layer_info:
    height = (n / max_n) * 4.5
    rect   = mpatches.FancyBboxPatch(
        (x - 0.35, (5 - height) / 2), 0.7, height,
        boxstyle='round,pad=0.05', facecolor=color, alpha=0.75, edgecolor='white', linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x, -0.3, label, ha='center', va='top', fontsize=9, color='#2c3e50')
    ax.text(x, (5 - height) / 2 + height / 2, str(n),
            ha='center', va='center', fontsize=11, fontweight='bold', color='white')

for i in range(len(layer_info) - 1):
    x1, x2 = layer_info[i][0] + 0.35, layer_info[i+1][0] - 0.35
    ax.annotate('', xy=(x2, 2.5), xytext=(x1, 2.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='#7f8c8d'))

ax.text(5, 5.3, 'Funnel ANN — [13 → 64 → 32 → 16 → 3]  with BatchNorm + Dropout(0.3)',
        ha='center', fontsize=12, fontweight='bold', color='#2c3e50')
plt.tight_layout(); plt.savefig('ann_funnel_diagram.png', dpi=150, bbox_inches='tight'); plt.show()
print(f'Funnel test accuracy: {results["Funnel"]["test"]:.3f}')

<a name="skip-code-8"></a>

**Expected output:** A color-coded bar diagram showing the shrinking layers from input to output. The visual makes the pyramid pattern immediately clear. Notice that each subsequent layer is half the width of the previous — this progressive compression forces the network to prioritize the most useful information at each stage.

**Accessibility note:** Add a Markdown cell describing the diagram: which layer is widest? What does the decreasing width represent?

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 2: Convolutional Neural Networks

## The Problem ANNs Cannot Solve

A fully connected ANN on images would need to flatten the image into a vector first. A 28×28 image becomes 784 numbers — and the network treats pixel 0 and pixel 783 as if they are equally related. Nearby pixels are not special. Spatial structure is invisible to a fully connected layer.

**CNNs solve this by learning local filters** — small learnable patterns that slide across the image and detect features wherever they appear.

## Key CNN Concepts

### Convolution

A **filter** (kernel) is a small matrix (e.g., 3×3) of learnable weights. The filter slides across the image, computing a dot product at each position. The result is a **feature map** — a spatial map of where that filter's pattern was detected.

```
Filter:    Feature map after convolution:
[1  0  -1]  [Strong positive = vertical edge going left]
[2  0  -2]  [Near zero       = no vertical edge]
[1  0  -1]  [Strong negative = vertical edge going right]
```

Early layers learn edges and textures. Later layers combine these into shapes. Even later layers recognize objects. This is the **feature hierarchy** from Part 1, now applied spatially.

### Pooling

**MaxPooling** takes the maximum value in each small region (e.g., 2×2). This shrinks the spatial dimensions by half while retaining the strongest detected features. Pooling makes the network **translation invariant** — the same object in a slightly different position gives the same response.

### The CNN Hidden Layer Pattern

```
Convolutional layers     → Feature detectors (spatial hidden layers)
MaxPooling               → Spatial compression
Flatten                  → Bridge from spatial to vector representation
Fully connected layers   → Classification (same as ANN Part 1)
```

Convolutional layers ARE hidden layers — they are just specialized for spatial data. The same sizing intuition applies: early conv layers have few filters (16–32), deeper conv layers have more (64–128–256) as they learn more complex features.

## Dataset: Fashion-MNIST

Fashion-MNIST contains 70,000 grayscale 28×28 images of 10 clothing categories: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot. It is harder than digit MNIST and more representative of real image classification challenges.

---

<a name="read-code-9"></a>

**Cell 9 — Load Fashion-MNIST**

This cell downloads Fashion-MNIST via torchvision, applies normalization (mean=0.5, std=0.5 to bring pixel values to [-1, 1]), and creates DataLoaders with batch size 64. The first call may take a minute to download.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

fashion_train = torchvision.datasets.FashionMNIST(
    root='./data', train=True,  download=True, transform=transform)
fashion_test  = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

fashion_train_dl = DataLoader(fashion_train, batch_size=64, shuffle=True,  num_workers=2)
fashion_test_dl  = DataLoader(fashion_test,  batch_size=64, shuffle=False, num_workers=2)

class_names = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

print(f'Training images:  {len(fashion_train)}')
print(f'Test images:      {len(fashion_test)}')
print(f'Image shape:      {fashion_train[0][0].shape}  (channels, height, width)')

# Show sample images
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = fashion_train[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[label], fontsize=7)
    ax.axis('off')
plt.suptitle('Fashion-MNIST Sample Images', fontsize=12, y=1.01)
plt.tight_layout(); plt.savefig('fashion_samples.png', dpi=150, bbox_inches='tight'); plt.show()

<a name="skip-code-9"></a>

**Expected output:** Dataset sizes (60,000 train, 10,000 test), image shape (1, 28, 28), and a grid of 16 sample images with labels. The (1, 28, 28) shape means: 1 color channel (grayscale), 28 pixels tall, 28 pixels wide.

**Accessibility note:** Add a Markdown cell describing the sample images — which categories look most similar to each other? Which would you expect a model to confuse?

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-10"></a>

**Cell 10 — Define the CNN Architecture**

This cell defines a CNN with two convolutional blocks followed by fully connected layers. The architecture annotation comments show the tensor shape at each step — tracing shapes through a CNN is an essential skill for debugging.

Shape calculation for conv + pool:
- Input: (N, 1, 28, 28)
- After Conv1(padding=1) + MaxPool(2): (N, 16, 14, 14)
- After Conv2(padding=1) + MaxPool(2): (N, 32, 7, 7)
- After Flatten: (N, 32×7×7) = (N, 1568)

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional blocks (spatial feature extraction)
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # (N,1,28,28) -> (N,16,28,28)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2)                              # (N,16,28,28) -> (N,16,14,14)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # (N,16,14,14) -> (N,32,14,14)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)                              # (N,32,14,14) -> (N,32,7,7)
        )

        # Fully connected classifier (same as ANN Part 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # (N,32,7,7) -> (N,1568)
            nn.Linear(1568, 256),         # Hidden layer 1
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),          # Hidden layer 2
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)            # Output: 10 classes (no Softmax here)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        return self.classifier(x)

cnn_model = FashionCNN().to(DEVICE)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f'CNN total parameters: {total_params:,}')
print(cnn_model)

<a name="skip-code-10"></a>

**Expected output:** Parameter count and the network architecture printout. Notice how the classifier at the end is identical in structure to the ANN from Part 1 — the conv layers are just specialized hidden layers that produce the input for the familiar fully-connected stack.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-11"></a>

**Cell 11 — Train the CNN**

This cell trains the CNN for 10 epochs. On GPU this takes 2–3 minutes; on CPU closer to 15–20 minutes. We use the same `train_model` function from Part 0.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)

print('Training CNN on Fashion-MNIST...')
cnn_history = train_model(
    cnn_model, fashion_train_dl, cnn_criterion, cnn_optimizer,
    epochs=10, desc='CNN'
)

test_acc = evaluate_model(cnn_model, fashion_test_dl)
print(f'\nFinal Test Accuracy: {test_acc:.4f}')
plot_loss(cnn_history, 'CNN Training Loss — Fashion-MNIST')

<a name="skip-code-11"></a>

**Expected output:** Loss decreasing over 10 epochs and final test accuracy around 88–91%. A simple ANN on flattened Fashion-MNIST images typically achieves ~84–87%. The CNN outperforms it because the convolutional layers exploit spatial structure that the flat ANN ignores.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-12"></a>

**Cell 12 — Visualize What the CNN Learned (Feature Maps)**

This cell passes a single image through the first conv block and visualizes the 16 feature maps it produces. Each feature map shows where the first layer's 16 learned filters activate most strongly. Some will detect edges; some will detect textures; some will seem to detect nothing obvious — early layers learn very low-level features.

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
cnn_model.eval()

# Get one test image
img_tensor, label = fashion_test[5]
print(f'Image label: {class_names[label]}')

# Pass through first conv block only
with torch.no_grad():
    x = img_tensor.unsqueeze(0).to(DEVICE)   # Add batch dimension: (1,1,28,28)
    feature_maps = cnn_model.conv_block1(x)  # (1,16,14,14)

feature_maps = feature_maps.squeeze(0).cpu()  # (16,14,14)

fig, axes = plt.subplots(3, 6, figsize=(14, 7))

# Original image in top-left
axes[0, 0].imshow(img_tensor.squeeze(), cmap='gray')
axes[0, 0].set_title(f'Original\n({class_names[label]})', fontsize=8)
axes[0, 0].axis('off')

# 16 feature maps
for i, ax in enumerate(axes.flat[1:17]):
    ax.imshow(feature_maps[i], cmap='viridis')
    ax.set_title(f'Filter {i+1}', fontsize=7)
    ax.axis('off')

# Hide unused axes
for ax in axes.flat[17:]:
    ax.axis('off')

plt.suptitle('First Conv Block — 16 Feature Maps\n(bright = strong activation)', fontsize=12)
plt.tight_layout(); plt.savefig('cnn_feature_maps.png', dpi=150, bbox_inches='tight'); plt.show()

<a name="skip-code-12"></a>

**Expected output:** The original image alongside 16 feature maps. Each map highlights a different aspect of the image — some filters will activate along edges, others on specific textures like the weave pattern in fabric. This visualization directly shows the feature hierarchy concept from Part 1: the first layer extracts basic visual features that subsequent layers combine into higher-level representations.

**Accessibility note:** Add a Markdown cell describing what you observe in the feature maps — which filters seem to be detecting edges? Textures? Are any hard to interpret?

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 3: Recurrent Neural Networks

## The Problem CNNs Cannot Solve

CNNs are excellent for spatial data but have no sense of order. A sentence, a time series, or a musical phrase is defined by its *sequence* — 'the cat sat on the mat' is different from 'the mat sat on the cat'. A CNN would treat these identically after flattening.

**RNNs solve this by maintaining a hidden state** that is updated at each step in the sequence. The hidden state acts as a rolling memory of everything the network has seen so far.

## How an RNN Works

At each time step $t$, the RNN receives an input $x_t$ and the previous hidden state $h_{t-1}$:

$$h_t = \tanh(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b)$$

The output at each step (or the final hidden state) is used for prediction:

$$y_t = W_{hy} \cdot h_t + b_y$$

```
         h₀ → [RNN] → h₁ → [RNN] → h₂ → [RNN] → h₃ → [output]
                ↑              ↑              ↑
               x₁             x₂             x₃
```

The same weight matrices ($W_{hh}$, $W_{xh}$) are used at every time step — the RNN 'unrolls' through time but shares parameters across steps.

## The Vanishing Gradient Problem

Training an RNN requires backpropagating gradients through time (BPTT). For a sequence of length 100, the gradient must travel through 100 tanh activations. Because tanh saturates (derivative ≤ 1), the gradient gets multiplied by a value ≤ 1 at each step. After 100 steps: $0.9^{100} \approx 0.000027$.

**The gradient vanishes** — the network cannot learn dependencies that span many steps. This is exactly the problem LSTM (Part 4) was designed to solve.

## Task: Sine Wave Next-Step Prediction

We will train an RNN to predict the next value in a sine wave — a simple, clean sequential task that clearly demonstrates both the power and the limitation of vanilla RNNs.

---

<a name="read-code-13"></a>

**Cell 13 — Generate the Sine Wave Dataset**

This cell generates a sine wave, splits it into overlapping windows of length 30 (the input sequence) with the next value as the target, and creates DataLoaders. This is the 'sliding window' approach from the Time Series notebook, now implemented for a sequence model rather than a lag-feature regressor.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Generate sine wave data
SEQ_LEN = 30    # Look back 30 steps
N_TRAIN  = 800

t = np.linspace(0, 16 * np.pi, 1500)
wave = np.sin(t).astype(np.float32)

# Create sliding windows
Xs, ys = [], []
for i in range(len(wave) - SEQ_LEN - 1):
    Xs.append(wave[i : i + SEQ_LEN])
    ys.append(wave[i + SEQ_LEN])

Xs = np.array(Xs, dtype=np.float32)  # (N, SEQ_LEN)
ys = np.array(ys, dtype=np.float32)  # (N,)

# Train/test split (no shuffling — sequential data)
X_sin_tr, X_sin_te = Xs[:N_TRAIN], Xs[N_TRAIN:]
y_sin_tr, y_sin_te = ys[:N_TRAIN], ys[N_TRAIN:]

# Reshape for RNN: (N, SEQ_LEN, INPUT_SIZE=1)
X_sin_tr_t = torch.tensor(X_sin_tr).unsqueeze(-1)  # (N, 30, 1)
X_sin_te_t = torch.tensor(X_sin_te).unsqueeze(-1)
y_sin_tr_t = torch.tensor(y_sin_tr)
y_sin_te_t = torch.tensor(y_sin_te)

sin_train_dl = DataLoader(TensorDataset(X_sin_tr_t, y_sin_tr_t), batch_size=64, shuffle=True)
sin_test_dl  = DataLoader(TensorDataset(X_sin_te_t, y_sin_te_t), batch_size=64)

print(f'Input shape: {X_sin_tr_t.shape}  (samples, sequence_length, features)')
print(f'Target shape: {y_sin_tr_t.shape}')

# Visualize the wave
plt.figure(figsize=(12, 3))
plt.plot(t[:200], wave[:200], color='steelblue', linewidth=2)
plt.axvline(t[N_TRAIN], color='red', linestyle='--', label='Train/Test split')
plt.title('Sine Wave — Input Sequence for RNN'); plt.xlabel('Time'); plt.ylabel('Value')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('sine_wave.png', dpi=150, bbox_inches='tight'); plt.show()

<a name="skip-code-13"></a>

**Expected output:** Tensor shapes and a sine wave plot. The input tensor has three dimensions: (batch, sequence_length, features). This 3D shape is required by all PyTorch sequence models (RNN, LSTM, GRU) — the last dimension is the input size at each time step (1, since we have one value per step).

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-14"></a>

**Cell 14 — Define and Train the RNN**

This cell defines an RNN model using `nn.RNN`. The RNN processes the sequence step by step, updating its hidden state each time. We use the final hidden state to predict the next value in the sequence. The output is a single number (regression, not classification), so we use `nn.MSELoss` instead of `CrossEntropyLoss`.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
class SineRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.rnn = nn.RNN(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,     # Input shape: (batch, seq, features)
            nonlinearity = 'tanh',
            dropout     = 0.2 if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)   # Final prediction head

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, h_n = self.rnn(x)    # out: (batch, seq_len, hidden)
        last_out = out[:, -1, :]  # Take the last time step: (batch, hidden)
        return self.fc(last_out).squeeze(-1)  # (batch,)

rnn_model = SineRNN(input_size=1, hidden_size=64, num_layers=2)
print(f'RNN parameters: {sum(p.numel() for p in rnn_model.parameters()):,}')

# Training — regression uses MSELoss
def train_regressor(model, loader, criterion, optimizer, epochs=30):
    model.train(); history = []
    for epoch in range(1, epochs + 1):
        total_loss = 0
        for Xb, yb in loader:
            optimizer.zero_grad()
            pred = model(Xb)
            loss = criterion(pred, yb)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        history.append(total_loss / len(loader))
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}  Loss: {history[-1]:.6f}')
    return history

rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=1e-3)
rnn_criterion = nn.MSELoss()
print('\nTraining RNN...')
rnn_history = train_regressor(rnn_model, sin_train_dl, rnn_criterion, rnn_optimizer, epochs=50)

<a name="skip-code-14"></a>

**Expected output:** Parameter count (~20K) and training loss decreasing toward a very small value. RNNs train quickly on this task because the sine wave is highly regular. Watch for the loss curve: it should decrease smoothly, indicating the RNN has learned the periodic pattern.

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-15"></a>

**Cell 15 — Evaluate RNN Predictions**

This cell generates predictions on the test set and overlays them on the actual sine wave. A well-trained RNN should track the wave closely on the near future but may drift on longer horizons — this is the vanishing gradient limitation made visible.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
rnn_model.eval()
with torch.no_grad():
    preds = rnn_model(X_sin_te_t).numpy()

actuals = y_sin_te_t.numpy()
rmse    = np.sqrt(np.mean((preds - actuals)**2))

plt.figure(figsize=(13, 4))
plt.plot(actuals[:200], label='Actual',   color='steelblue', linewidth=2)
plt.plot(preds[:200],   label='Predicted', color='darkorange', linewidth=1.5, linestyle='--')
plt.xlabel('Time Step'); plt.ylabel('Value')
plt.title(f'RNN Sine Wave Predictions (Test Set)  |  RMSE = {rmse:.5f}')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('rnn_predictions.png', dpi=150, bbox_inches='tight'); plt.show()
print(f'Test RMSE: {rmse:.6f}')

<a name="skip-code-15"></a>

**Expected output:** A close match between actual and predicted values, with RMSE typically below 0.05. The sine wave is periodic and smooth, so even a vanilla RNN handles it reasonably well. The limitation of vanilla RNNs becomes apparent on longer sequences — try increasing `SEQ_LEN` to 100 and rerunning: the RNN will struggle to maintain accuracy because its gradient must flow through 100 tanh activations during training. This motivates LSTM.

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 4: Long Short-Term Memory (LSTM)

## The Problem RNNs Cannot Solve

Vanilla RNNs suffer from **vanishing gradients** over long sequences. The hidden state must carry information from step 1 all the way to step 100 — but each step multiplies the gradient by the tanh derivative (≤ 1), causing the gradient to approach zero exponentially.

The result: the network forgets early context. In the sentence *'The cat, which spent three months in the shelter before being adopted, finally found its **home**'*, an RNN may struggle to connect 'home' to 'cat' across all the intervening words.

## LSTM's Solution: Gated Memory

LSTMs introduce a **cell state** $c_t$ — a 'conveyor belt' that runs alongside the hidden state and can carry information across many steps with minimal modification. Three gates control what enters, what is forgotten, and what is output:

| Gate | Symbol | Formula | Purpose |
| :--- | :--- | :--- | :--- |
| **Forget gate** | $f_t$ | $\sigma(W_f [h_{t-1}, x_t] + b_f)$ | What to erase from cell state |
| **Input gate** | $i_t$ | $\sigma(W_i [h_{t-1}, x_t] + b_i)$ | What new info to write to cell state |
| **Output gate** | $o_t$ | $\sigma(W_o [h_{t-1}, x_t] + b_o)$ | What to expose from cell state |

The cell state update:

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

Because the cell state is updated by **addition** (not multiplication through tanh), gradients can flow backwards through many steps without vanishing. The forget gate learning to stay near 1 means 'remember this' — the gradient flows through a multiplication by ~1 rather than ~0.3.

## Task: Sentiment Classification

We will train an LSTM to classify short movie reviews as positive or negative. This task requires the model to integrate information across an entire sentence — exactly where LSTMs shine over vanilla RNNs.

---

<a name="read-code-16"></a>

**Cell 16 — Prepare a Small Sentiment Dataset**

This cell creates a hand-crafted sentiment dataset of 80 labeled sentences, builds a word-level vocabulary, converts sentences to padded integer sequences, and wraps them in DataLoaders. The vocabulary and padding techniques here are the same used in production NLP systems — just scaled down.

<nav aria-label="Code cell 16 navigation">
<a href="#skip-code-16" aria-label="Skip code cell 16 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Small sentiment corpus
POS = [
    'this film was absolutely wonderful and moving',
    'the acting and direction were both superb',
    'a masterpiece of modern cinema',
    'brilliantly crafted and deeply emotional',
    'one of the best films I have seen this year',
    'the story was captivating from start to finish',
    'outstanding performances from the entire cast',
    'a beautiful and thought-provoking experience',
    'the cinematography was breathtaking and gorgeous',
    'I was completely engrossed throughout',
    'funny touching and utterly charming',
    'a delightful surprise that exceeded all expectations',
    'the script was sharp intelligent and witty',
    'heartfelt performances that moved me to tears',
    'a genuinely excellent film in every way',
    'refreshing original and beautifully executed',
    'the pacing was perfect and kept me hooked',
    'a wonderful tribute to human resilience',
    'clever sophisticated and deeply satisfying',
    'the best movie I have watched in years',
] * 2

NEG = [
    'this was a complete waste of my time',
    'terribly boring and completely predictable',
    'the worst film I have ever seen',
    'painfully slow and deeply disappointing',
    'the acting was absolutely dreadful',
    'nothing works in this disaster of a film',
    'a confusing and poorly written mess',
    'I fell asleep halfway through',
    'the plot made absolutely no sense',
    'an embarrassing failure in every way',
    'dull uninspired and forgettable',
    'the dialogue was cringe-worthy and awful',
    'a boring slog that goes nowhere',
    'completely hollow and emotionally empty',
    'I regret every minute I spent watching this',
    'terrible direction and atrocious editing',
    'a poorly executed and deeply frustrating film',
    'meandering pointless and painfully dull',
    'the worst script I have encountered',
    'a shameless cash grab with no redeeming qualities',
] * 2

sentences = [(s, 1) for s in POS] + [(s, 0) for s in NEG]
random.shuffle(sentences)

# Build vocabulary
all_words = set()
for s, _ in sentences:
    all_words.update(s.lower().split())
vocab     = {w: i+2 for i, w in enumerate(sorted(all_words))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
VOCAB_SIZE = len(vocab)

MAX_LEN = 12
def encode(sentence):
    tokens = [vocab.get(w, 1) for w in sentence.lower().split()]
    if len(tokens) < MAX_LEN:
        tokens += [0] * (MAX_LEN - len(tokens))
    return tokens[:MAX_LEN]

X_sent = torch.tensor([encode(s) for s, _ in sentences], dtype=torch.long)
y_sent = torch.tensor([y for _, y in sentences], dtype=torch.long)

X_s_tr, X_s_te, y_s_tr, y_s_te = train_test_split(
    X_sent, y_sent, test_size=0.25, random_state=42)

sentiment_train_dl = DataLoader(TensorDataset(X_s_tr, y_s_tr), batch_size=16, shuffle=True)
sentiment_test_dl  = DataLoader(TensorDataset(X_s_te, y_s_te), batch_size=16)
print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'Training samples: {len(X_s_tr)}  |  Test: {len(X_s_te)}')
print(f'Sample encoding: {encode("this film was absolutely wonderful")}')

<a name="skip-code-16"></a>

**Expected output:** Vocabulary size, dataset sizes, and a sample integer encoding. Each word is converted to a unique integer. The sequence is padded with zeros to `MAX_LEN`. These integer sequences feed into an **Embedding layer** in the LSTM, which converts each integer into a dense learned vector (the word embedding).

<nav aria-label="Return navigation for code cell 16">
<a href="#read-code-16" aria-label="Go back and read code cell 16">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-17"></a>

**Cell 17 — Define and Train the Sentiment LSTM**

This cell defines an LSTM with an Embedding layer (converts words to learned vectors), two LSTM layers for deeper sequential processing, and a linear classifier on the final hidden state. The Embedding layer is itself a form of hidden layer — it learns a 32-dimensional representation for each word.

<nav aria-label="Code cell 17 navigation">
<a href="#skip-code-17" aria-label="Skip code cell 17 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_layers=2, n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = 0.3 if num_layers > 1 else 0,
            bidirectional = False
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        # x: (batch, seq_len) — integer token IDs
        emb = self.embedding(x)          # (batch, seq_len, embed_dim)
        out, (h_n, c_n) = self.lstm(emb) # h_n: (num_layers, batch, hidden)
        last_h = h_n[-1]                 # Take last layer's hidden state
        return self.classifier(last_h)

lstm_model     = SentimentLSTM(VOCAB_SIZE)
lstm_criterion = nn.CrossEntropyLoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=5e-3)

print(f'LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')
print('\nTraining LSTM...')
lstm_history = train_model(
    lstm_model, sentiment_train_dl, lstm_criterion, lstm_optimizer,
    epochs=60, desc='LSTM'
)

train_acc = evaluate_model(lstm_model, sentiment_train_dl)
test_acc  = evaluate_model(lstm_model, sentiment_test_dl)
print(f'\nTrain accuracy: {train_acc:.3f}  |  Test accuracy: {test_acc:.3f}')
plot_loss(lstm_history, 'LSTM Training Loss — Sentiment')

<a name="skip-code-17"></a>

**Expected output:** Parameter count and training loss curve. On this small dataset, the LSTM should achieve high train accuracy quickly. The key architectural insight: the LSTM takes word embeddings as input (32-dimensional vectors learned for each word) and processes them in sequence, accumulating a hidden state that represents the 'meaning so far' at each position. The final hidden state captures the meaning of the entire sentence.

<nav aria-label="Return navigation for code cell 17">
<a href="#read-code-17" aria-label="Go back and read code cell 17">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-18"></a>

**Cell 18 — Run the LSTM on New Sentences**

This cell demonstrates the trained LSTM on new sentences not in the training data. It shows the predicted class probabilities, letting you see the model's confidence level.

<nav aria-label="Code cell 18 navigation">
<a href="#skip-code-18" aria-label="Skip code cell 18 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
lstm_model.eval()

def predict_sentiment(sentences, model, vocab, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        X = torch.tensor([encode(s) for s in sentences], dtype=torch.long)
        logits = model(X)
        probs  = torch.softmax(logits, dim=1).numpy()
    for s, p in zip(sentences, probs):
        label = 'POSITIVE' if p[1] > 0.5 else 'NEGATIVE'
        print(f'  [{label}] conf={max(p):.2f}  "{s}"')

new_sentences = [
    'this was an absolutely wonderful experience',
    'boring and completely disappointing waste of time',
    'a surprisingly decent film with good performances',
    'terrible script but the visuals were stunning',
    'one of the best stories I have seen in years',
    'I regret watching this dull and pointless film',
]
print('=== Sentiment Predictions ===')
predict_sentiment(new_sentences, lstm_model, vocab)

<a name="skip-code-18"></a>

**Expected output:** Sentiment predictions for each sentence with confidence scores. The model should correctly classify the clearly positive and negative sentences. Mixed sentences like 'terrible script but the visuals were stunning' are interesting — how the model handles these reveals what it learned to weight most heavily. On a small training set, domain-specific vocabulary may not generalize perfectly.

<nav aria-label="Return navigation for code cell 18">
<a href="#read-code-18" aria-label="Go back and read code cell 18">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 5: Sequence-to-Sequence Models

## The Problem LSTMs Cannot Solve (Alone)

LSTMs can classify sequences — but what if your output is also a variable-length sequence? **Machine translation** (English → French), **text summarization** (article → summary), and **speech recognition** (audio → text) all require mapping one sequence to another.

A single LSTM cannot handle this directly because its output must be the same length as its input.

## The Encoder-Decoder Architecture

**Seq2Seq** (Sutskever et al., 2014) separates the problem into two steps:

```
Input sequence       Encoder           Context vector       Decoder         Output sequence
[1, 2, 3, 4, 5] →  [LSTM] →           [h₅, c₅]         →  [LSTM] →   [5, 4, 3, 2, 1]
                   step by step        (the 'thought          step by
                   compression         vector')              step generation
```

**The Encoder** reads the full input sequence and compresses it into a fixed-size **context vector** (the final hidden state). This single vector must represent the entire input meaning.

**The Decoder** starts from the context vector and generates the output sequence one token at a time. At each step, it takes the previous output as input and updates its hidden state.

## The Bottleneck Problem

The context vector is a bottleneck: no matter how long the input sequence is, it must be compressed into a fixed-size vector. For long sequences, information is lost. In a paper translating 100-word sentences, the encoder's final hidden state cannot faithfully represent every word.

**This is the problem the Attention Mechanism (Part 6) was invented to solve.**

## Task: Sequence Reversal

We train a Seq2Seq model to reverse integer sequences: `[3, 1, 4, 1, 5]` → `[5, 1, 4, 1, 3]`. This is a clean task that tests whether the encoder-decoder architecture truly works — reversing requires the decoder to selectively access information from all positions in the input.

---

<a name="read-code-19"></a>

**Cell 19 — Generate Sequence Reversal Data**

This cell generates 4,000 random integer sequences of length 7 and their reversed versions. The vocabulary is integers 1–9 plus special tokens: 0 for padding, 10 for Start-of-Sequence (SOS), and 11 for End-of-Sequence (EOS). The decoder needs SOS to know when to start generating and EOS to know when to stop.

<nav aria-label="Code cell 19 navigation">
<a href="#skip-code-19" aria-label="Skip code cell 19 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Special tokens
PAD, SOS, EOS = 0, 10, 11
VOCAB_S2S = 12     # 0-9 (real) + SOS + EOS
SEQ_LEN_S2S = 7
N_SAMPLES = 4000

# Generate data
seqs    = np.random.randint(1, 10, size=(N_SAMPLES, SEQ_LEN_S2S))
revseqs = np.flip(seqs, axis=1).copy()

# Add SOS and EOS to targets
# Decoder input:  [SOS, rev[0], rev[1], ..., rev[N-1]]
# Decoder target: [rev[0], rev[1], ..., rev[N-1], EOS]
dec_input  = np.hstack([np.full((N_SAMPLES, 1), SOS), revseqs])
dec_target = np.hstack([revseqs, np.full((N_SAMPLES, 1), EOS)])

# Convert to tensors
enc_in  = torch.tensor(seqs, dtype=torch.long)
dec_in  = torch.tensor(dec_input, dtype=torch.long)
dec_tgt = torch.tensor(dec_target, dtype=torch.long)

X_s2s_tr, X_s2s_te, di_tr, di_te, dt_tr, dt_te = train_test_split(
    enc_in, dec_in, dec_tgt, test_size=0.2, random_state=42)

s2s_train_dl = DataLoader(TensorDataset(X_s2s_tr, di_tr, dt_tr), batch_size=64, shuffle=True)
s2s_test_dl  = DataLoader(TensorDataset(X_s2s_te, di_te, dt_te), batch_size=64)

print(f'Encoder input shape:  {X_s2s_tr.shape}')
print(f'Decoder input shape:  {di_tr.shape}   (SOS + reversed sequence)')
print(f'Decoder target shape: {dt_tr.shape}   (reversed sequence + EOS)')
print(f'\nExample:')
print(f'  Encoder input:  {seqs[0].tolist()}')
print(f'  Decoder input:  {dec_input[0].tolist()}')
print(f'  Decoder target: {dec_target[0].tolist()}')

<a name="skip-code-19"></a>

**Expected output:** Tensor shapes and an example input/output pair. The decoder input starts with SOS (10) and the decoder target ends with EOS (11). This 'shifted by one' design is called **teacher forcing**: during training, the decoder receives the ground-truth previous token as input, making training faster and more stable.

<nav aria-label="Return navigation for code cell 19">
<a href="#read-code-19" aria-label="Go back and read code cell 19">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-20"></a>

**Cell 20 — Define the Encoder, Decoder, and Seq2Seq Model**

Three classes are defined. The Encoder reads the input and returns the context vector (hidden + cell states). The Decoder takes the context vector and generates output one token at a time. The Seq2Seq model connects them using teacher forcing during training.

<nav aria-label="Code cell 20 navigation">
<a href="#skip-code-20" aria-label="Skip code cell 20 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)             # (batch, seq, embed)
        _, (hidden, cell) = self.lstm(emb)  # Ignore output; keep final states
        return hidden, cell                 # Context vector


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        # x: (batch,) — single token ID at each step
        x   = x.unsqueeze(1)              # (batch, 1)
        emb = self.embedding(x)           # (batch, 1, embed)
        out, (hidden, cell) = self.lstm(emb, (hidden, cell))
        logits = self.fc(out.squeeze(1))  # (batch, vocab_size)
        return logits, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, vocab_size):
        super().__init__()
        self.encoder   = encoder
        self.decoder   = decoder
        self.vocab_size = vocab_size

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (batch, src_len)   trg: (batch, trg_len)
        batch_size, trg_len = trg.shape
        outputs = torch.zeros(batch_size, trg_len, self.vocab_size)

        hidden, cell = self.encoder(src)   # Encode input
        dec_input    = trg[:, 0]           # First decoder input = SOS

        for t in range(1, trg_len):
            logits, hidden, cell = self.decoder(dec_input, hidden, cell)
            outputs[:, t, :]    = logits
            # Teacher forcing: use true token or predicted token
            use_teacher  = random.random() < teacher_forcing_ratio
            dec_input    = trg[:, t] if use_teacher else logits.argmax(dim=1)

        return outputs

# Assemble
EMBED_DIM, HIDDEN_S2S, N_LAYERS_S2S = 32, 128, 2
enc = Encoder(VOCAB_S2S, EMBED_DIM, HIDDEN_S2S, N_LAYERS_S2S)
dec = Decoder(VOCAB_S2S, EMBED_DIM, HIDDEN_S2S, N_LAYERS_S2S)
s2s_model = Seq2Seq(enc, dec, VOCAB_S2S)
print(f'Seq2Seq parameters: {sum(p.numel() for p in s2s_model.parameters()):,}')

<a name="skip-code-20"></a>

**Expected output:** Parameter count for the full Seq2Seq model. Notice that the Encoder and Decoder share the same hidden size — the context vector produced by the Encoder must match the hidden size the Decoder expects. This shape constraint is the bottleneck: everything the Encoder learned must fit inside a vector of size 128.

<nav aria-label="Return navigation for code cell 20">
<a href="#read-code-20" aria-label="Go back and read code cell 20">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-21"></a>

**Cell 21 — Train the Seq2Seq Model**

Training Seq2Seq requires a custom loop because the model takes three inputs (encoder input, decoder input, decoder target) rather than the standard two. We clip gradients to prevent exploding gradients — a common issue in sequence models where errors accumulate over many steps.

<nav aria-label="Code cell 21 navigation">
<a href="#skip-code-21" aria-label="Skip code cell 21 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
s2s_optimizer = optim.Adam(s2s_model.parameters(), lr=5e-3)
s2s_criterion = nn.CrossEntropyLoss(ignore_index=PAD)

def train_s2s(model, loader, optimizer, criterion, epochs=40):
    model.train(); history = []
    for epoch in range(1, epochs + 1):
        total_loss = 0
        for src, trg_in, trg_out in loader:
            optimizer.zero_grad()
            outputs = model(src, trg_in)  # (batch, trg_len, vocab)
            # Reshape for CrossEntropyLoss: (batch*seq, vocab) vs (batch*seq,)
            loss = criterion(
                outputs[:, 1:, :].reshape(-1, model.vocab_size),
                trg_out[:, 1:].reshape(-1)
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Prevent exploding grad
            optimizer.step()
            total_loss += loss.item()
        history.append(total_loss / len(loader))
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}  Loss: {history[-1]:.4f}')
    return history

print('Training Seq2Seq...')
s2s_history = train_s2s(s2s_model, s2s_train_dl, s2s_optimizer, s2s_criterion, epochs=50)
plot_loss(s2s_history, 'Seq2Seq Training Loss — Sequence Reversal')

<a name="skip-code-21"></a>

**Expected output:** Loss decreasing from ~2.5 toward ~0.1 over 50 epochs. Sequence reversal requires the decoder to generate each token in reverse order — it must selectively recall position information from the context vector. If your loss plateau above 0.3, try increasing HIDDEN_S2S or training longer.

<nav aria-label="Return navigation for code cell 21">
<a href="#read-code-21" aria-label="Go back and read code cell 21">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-22"></a>

**Cell 22 — Test the Seq2Seq Model: Can It Reverse?**

This cell runs the trained model on new sequences in inference mode — no teacher forcing, just the model generating each token from its own previous output. This is how the model would behave in production.

<nav aria-label="Code cell 22 navigation">
<a href="#skip-code-22" aria-label="Skip code cell 22 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
s2s_model.eval()

def reverse_sequence(model, seq, device=DEVICE):
    """Use the Seq2Seq model to reverse a sequence."""
    model.eval()
    with torch.no_grad():
        src = torch.tensor(seq).unsqueeze(0)   # (1, seq_len)
        hidden, cell = model.encoder(src)
        dec_input    = torch.tensor([SOS])     # Start with SOS
        result       = []
        for _ in range(len(seq) + 1):          # Max output length
            logits, hidden, cell = model.decoder(dec_input, hidden, cell)
            pred = logits.argmax(dim=1)
            if pred.item() == EOS:
                break
            result.append(pred.item())
            dec_input = pred
    return result

# Test on 8 examples
print('=== Sequence Reversal Test ===')
print(f'{"Input":20s}  {"Target":20s}  {"Predicted":20s}  Match')
for _ in range(8):
    seq    = np.random.randint(1, 10, size=SEQ_LEN_S2S).tolist()
    target = list(reversed(seq))
    pred   = reverse_sequence(s2s_model, seq)
    match  = '✓' if pred == target else '✗'
    print(f'{str(seq):20s}  {str(target):20s}  {str(pred):20s}  {match}')

<a name="skip-code-22"></a>

**Expected output:** A table showing input sequences, the correct reversed output, and the model's prediction. A well-trained model should achieve close to 100% accuracy on this task. If it makes errors on longer sequences, this demonstrates the bottleneck problem — the context vector must encode all position information, which becomes harder as length grows. This is precisely what Attention (Part 6) solves.

<nav aria-label="Return navigation for code cell 22">
<a href="#read-code-22" aria-label="Go back and read code cell 22">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 6: The Attention Mechanism

## The Problem Seq2Seq Cannot Solve

The Seq2Seq encoder compresses the entire input into a single fixed-size context vector. For long sequences this bottleneck causes information loss — it is like summarizing a 100-page document into one sentence and then asking someone to answer detailed questions from that summary.

**Attention** (Bahdanau et al., 2015) solves this by giving the decoder direct access to all encoder hidden states — not just the final one. At each decoding step, the decoder learns to **attend** to the most relevant parts of the input.

## The Attention Intuition

Imagine translating *'The cat sat on the mat'* into French. When generating the French word for 'cat', the decoder should look at the encoder state that processed 'cat'. When generating the word for 'mat', it should look at the encoder state that processed 'mat'. Attention learns this alignment automatically.

## Scaled Dot-Product Attention

The Transformer (Part 7) uses **Scaled Dot-Product Attention**, which maps three inputs to an output:

$$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q (Query):** What I am looking for (decoder's current state)
- **K (Key):** What each encoder state 'announces' (encoder hidden states, indexed)
- **V (Value):** What each encoder state 'contains' (the actual information to retrieve)

The dot product $QK^T$ computes a **similarity score** between the query and each key. Softmax converts these scores into **attention weights** (a probability distribution). The weighted sum of Values is the **context vector** — a blend of encoder states weighted by relevance to the current decoding step.

The $\frac{1}{\sqrt{d_k}}$ scaling prevents the dot products from growing too large in high dimensions, which would push softmax into regions with near-zero gradients.

## From Bahdanau to Transformer Attention

| Version | Year | Key Difference |
| :--- | :--- | :--- |
| Bahdanau (Additive) | 2015 | Trains a small network to compute alignment scores |
| Luong (Multiplicative) | 2015 | Dot product between decoder state and all encoder states |
| Scaled Dot-Product | 2017 | Luong attention scaled by $\sqrt{d_k}$, generalized to Q/K/V |

---

<a name="read-code-23"></a>

**Cell 23 — Scaled Dot-Product Attention from Scratch**

This cell implements the attention formula as a standalone function, then demonstrates it on random Q, K, V matrices. Implementing this yourself — rather than just calling PyTorch's version — is the fastest way to genuinely understand what the Transformer is doing.

<nav aria-label="Code cell 23 navigation">
<a href="#skip-code-23" aria-label="Skip code cell 23 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention.
    Q: (batch, heads, seq_q, d_k)
    K: (batch, heads, seq_k, d_k)
    V: (batch, heads, seq_k, d_v)
    """
    d_k = Q.shape[-1]

    # Step 1: Compute similarity scores  (batch, heads, seq_q, seq_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2: Apply mask (if provided — used for causal/padding masking)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 3: Softmax over key dimension → attention weights
    attn_weights = torch.softmax(scores, dim=-1)

    # Step 4: Weighted sum of Values
    context = torch.matmul(attn_weights, V)

    return context, attn_weights

# Demonstrate on a mini example
batch, seq_len, d_k = 1, 5, 16
Q = torch.randn(batch, 1, seq_len, d_k)
K = torch.randn(batch, 1, seq_len, d_k)
V = torch.randn(batch, 1, seq_len, d_k)

context, weights = scaled_dot_product_attention(Q, K, V)

print('=== Attention Shapes ===')
print(f'Q shape:              {Q.shape}          (batch, heads, seq_len, d_k)')
print(f'Attention weights:    {weights.shape}    (batch, heads, seq_q, seq_k)')
print(f'Context vector:       {context.shape}    (batch, heads, seq_q, d_k)')
print(f'\nAttention weights (row = query position, col = key position):')
print(weights[0, 0].detach().numpy().round(3))
print(f'\nRow sums (should all be 1.0): {weights[0, 0].sum(dim=-1).detach().numpy().round(4)}')

<a name="skip-code-23"></a>

**Expected output:** The shapes at each step and the attention weight matrix. Each row of the attention weight matrix is a probability distribution — it sums to 1.0. Each entry `[i, j]` represents how much query position `i` attends to key position `j`. A weight of 0.4 at position [2, 4] means: when generating the output for position 2, 40% of the information comes from position 4's value.

<nav aria-label="Return navigation for code cell 23">
<a href="#read-code-23" aria-label="Go back and read code cell 23">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-24"></a>

**Cell 24 — Visualize Attention Weights as a Heatmap**

This cell creates a synthetic 'perfect reversal' attention pattern to show what ideal attention looks like for the sequence reversal task. To reverse `[3, 1, 4, 1, 5]` → `[5, 1, 4, 1, 3]`, the decoder at step 1 (generating 5) should attend to encoder position 5 (the last element). The ideal attention matrix is the anti-diagonal.

<nav aria-label="Code cell 24 navigation">
<a href="#skip-code-24" aria-label="Skip code cell 24 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Simulate ideal attention for sequence reversal
# Position i in output should attend to position (n-1-i) in input
n = 7
ideal_attn = np.zeros((n, n))
for i in range(n):
    ideal_attn[i, n - 1 - i] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Ideal attention
im1 = axes[0].imshow(ideal_attn, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im1, ax=axes[0])
axes[0].set_xlabel('Encoder Position (Input)'); axes[0].set_ylabel('Decoder Step (Output)')
axes[0].set_title('Ideal Attention — Perfect Reversal\n(anti-diagonal pattern)', fontsize=11)
axes[0].set_xticks(range(n)); axes[0].set_yticks(range(n))

# Right: Random/learned attention (demonstration)
np.random.seed(7)
learned_attn = np.exp(np.random.randn(n, n) * 1.5)
learned_attn /= learned_attn.sum(axis=1, keepdims=True)

im2 = axes[1].imshow(learned_attn, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im2, ax=axes[1])
axes[1].set_xlabel('Encoder Position (Input)'); axes[1].set_ylabel('Decoder Step (Output)')
axes[1].set_title('Learned Attention Pattern\n(approximate — varies by training)', fontsize=11)
axes[1].set_xticks(range(n)); axes[1].set_yticks(range(n))

plt.suptitle('Attention Weight Matrices — Sequence Reversal', fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig('attention_heatmaps.png', dpi=150, bbox_inches='tight'); plt.show()

print('The attention weight matrix is the core interpretability tool of Transformers.')
print('Each row = one output step.  Each column = one input position.')
print('High weight = strong attention = important source of information.')

<a name="skip-code-24"></a>

**Expected output:** Two heatmaps — ideal and approximate learned attention. The ideal (anti-diagonal) pattern shows exactly what attention should learn for reversal: output step 0 attends to input position 6, output step 1 attends to position 5, and so on. This visualization is one of the most powerful interpretability tools in deep learning — you can literally see what the model is looking at when making each prediction.

<nav aria-label="Return navigation for code cell 24">
<a href="#read-code-24" aria-label="Go back and read code cell 24">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 7: The Transformer
# 'Attention Is All You Need' (Vaswani et al., 2017)

## The Core Claim

> *'We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.'*

Before 2017, the best sequence models combined RNNs with attention. The Transformer's radical claim was: you do not need the RNN at all. Attention alone is sufficient — and better.

## Why Recurrence Was a Problem

RNNs and LSTMs process sequences **sequentially** — step 5 cannot start until step 4 finishes. This makes training on long sequences slow because it cannot be parallelized. A 512-token sequence requires 512 sequential LSTM steps.

The Transformer processes **all tokens simultaneously** using self-attention. Every position attends to every other position in a single matrix operation. This is the key that unlocked training on massive datasets.

## Key Contributions of the 2017 Paper

### 1. Self-Attention

In Seq2Seq attention, the query came from the decoder and keys/values from the encoder. **Self-attention** applies the same mechanism *within* a single sequence: each position attends to all other positions in the same sequence.

This gives every position a global view of the full context — word 1 can directly attend to word 100 in a single step, with no gradient decay across the 99 intervening positions.

### 2. Multi-Head Attention

Instead of one attention computation, the Transformer runs $h$ attention heads in parallel, each with different learned Q, K, V projections:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

Different heads learn to attend to different types of relationships simultaneously — one head might track syntactic structure, another semantic similarity, another coreference.

### 3. Positional Encoding

Self-attention has no inherent sense of position — [A, B, C] and [C, B, A] produce the same set of attention scores if the contents match. The Transformer injects **positional encodings** — sine and cosine functions of position — into the token embeddings so the model knows where each token is:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

### 4. The Feed-Forward Sublayer

After attention, each position passes through a small fully-connected network (same weights applied independently at each position):

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

This is the network's opportunity to process the attended information non-linearly at each position. The hidden dimension of the FFN (typically 4× the model dimension) is where most of the parameters in a Transformer live.

### 5. Residual Connections and Layer Normalization

Each sublayer (attention or FFN) is wrapped in:

$$\text{output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

The residual connection (`x + ...`) allows gradients to flow through the network unimpeded — the gradient can bypass any sublayer via the shortcut. This is what allows Transformers to be trained at depths of 12, 24, or even 96 layers.

## The Transformer Architecture

```
     Input Tokens
          ↓
     Embedding + Positional Encoding
          ↓
  ┌─────────────────────────────────┐
  │  Multi-Head Self-Attention      │
  │  Add & Norm (residual)          │  ×N layers
  │  Feed-Forward Network           │
  │  Add & Norm (residual)          │
  └─────────────────────────────────┘
          ↓
     Pool / [CLS] token
          ↓
     Linear Classifier
          ↓
     Output Predictions
```

---

<a name="read-code-25"></a>

**Cell 25 — Positional Encoding**

This cell implements the sinusoidal positional encoding from the paper and visualizes it as a heatmap. Each row is a position (0 to max_len-1), and each column is one dimension of the encoding. The sinusoidal pattern means similar positions have similar encodings, and the model can generalize to positions it has not seen during training.

<nav aria-label="Code cell 25 navigation">
<a href="#skip-code-25" aria-label="Skip code cell 25 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Build the (max_len, d_model) positional encoding matrix
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()          # (max_len, 1)
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))             # (d_model/2,)
        pe[:, 0::2] = torch.sin(pos * div)   # Even dimensions: sine
        pe[:, 1::2] = torch.cos(pos * div)   # Odd dimensions: cosine
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return self.dropout(x + self.pe[:, :x.size(1), :])

# Visualize
d_model, max_len = 64, 50
pe_module = PositionalEncoding(d_model=d_model, max_len=max_len)
pe_matrix = pe_module.pe[0].detach().numpy()  # (50, 64)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(pe_matrix, cmap='RdBu', aspect='auto')
axes[0].set_xlabel('Embedding Dimension'); axes[0].set_ylabel('Position in Sequence')
axes[0].set_title('Sinusoidal Positional Encoding Heatmap\n(red=+1, blue=-1)')
plt.colorbar(axes[0].images[0], ax=axes[0])

# Show first 4 dimensions as sine curves
for i in range(4):
    axes[1].plot(pe_matrix[:, i], label=f'Dim {i}', linewidth=1.5)
axes[1].set_xlabel('Position'); axes[1].set_ylabel('Encoding Value')
axes[1].set_title('First 4 Dimensions of Positional Encoding\n(alternating sine and cosine)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('positional_encoding.png', dpi=150, bbox_inches='tight'); plt.show()
print('Positional encoding defined and visualized.')

<a name="skip-code-25"></a>

**Expected output:** A heatmap showing the full positional encoding matrix and a line plot of the first four dimensions. Each row is a unique fingerprint for a sequence position. Nearby positions have similar fingerprints (similar sine values), and the model can compute relative positions by comparing them. The different frequencies (low frequency in early dims, high frequency in later dims) give the model a sense of both coarse and fine-grained position information.

<nav aria-label="Return navigation for code cell 25">
<a href="#read-code-25" aria-label="Go back and read code cell 25">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-26"></a>

**Cell 26 — Build a Transformer Encoder for Text Classification**

This cell builds a complete Transformer classifier using PyTorch's built-in `nn.TransformerEncoderLayer` and `nn.TransformerEncoder`. We apply it to the sentiment classification task from Part 4, giving us a direct comparison: LSTM vs Transformer on the same problem.

<nav aria-label="Code cell 26 navigation">
<a href="#skip-code-26" aria-label="Skip code cell 26 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_len=MAX_LEN, n_classes=2, dropout=0.2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)

        encoder_layer  = nn.TransformerEncoderLayer(
            d_model     = d_model,    # Model dimension
            nhead       = n_heads,    # Number of attention heads
            dim_feedforward = d_ff,  # Hidden size of FFN sublayer (typically 4x d_model)
            dropout     = dropout,
            batch_first = True        # Input: (batch, seq, d_model)
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Classifier head: mean pool across sequence, then linear
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        # x: (batch, seq_len) — integer token IDs
        x = self.embedding(x)              # (batch, seq, d_model)
        x = self.pos_enc(x)                # Add positional information
        x = self.transformer_encoder(x)    # Self-attention + FFN × n_layers
        x = x.mean(dim=1)                  # Mean pool across sequence
        return self.classifier(x)

transformer_model = TransformerClassifier(
    vocab_size=VOCAB_SIZE, d_model=64, n_heads=4, n_layers=2, d_ff=256
)
total_params = sum(p.numel() for p in transformer_model.parameters())
print(f'Transformer parameters: {total_params:,}')

# Parameter breakdown
for name, module in transformer_model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f'  {name:30s}: {params:,}')

<a name="skip-code-26"></a>

**Expected output:** Total parameter count and breakdown by component. Notice that the `transformer_encoder` holds most of the parameters — the multi-head attention and feed-forward sublayers within it. The classifier head is the same small ANN from Part 1, applied after the Transformer has produced a contextually enriched representation of each token.

<nav aria-label="Return navigation for code cell 26">
<a href="#read-code-26" aria-label="Go back and read code cell 26">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-27"></a>

**Cell 27 — Train the Transformer and Compare to LSTM**

This cell trains the Transformer on the same sentiment dataset as the LSTM and prints a direct comparison. On this tiny dataset, both models should perform similarly — the Transformer's advantage over LSTMs appears at scale (larger datasets, longer sequences, deeper models).

<nav aria-label="Code cell 27 navigation">
<a href="#skip-code-27" aria-label="Skip code cell 27 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
transformer_criterion = nn.CrossEntropyLoss()
transformer_optimizer = optim.Adam(transformer_model.parameters(), lr=1e-3, weight_decay=1e-4)

print('Training Transformer...')
transformer_history = train_model(
    transformer_model, sentiment_train_dl, transformer_criterion,
    transformer_optimizer, epochs=80, desc='Transformer'
)

lstm_test_acc = evaluate_model(lstm_model, sentiment_test_dl)
trans_test_acc = evaluate_model(transformer_model, sentiment_test_dl)

print('\n=== Model Comparison: Sentiment Classification ===')
print(f'  LSTM Test Accuracy:        {lstm_test_acc:.3f}')
print(f'  Transformer Test Accuracy: {trans_test_acc:.3f}')

# Plot both learning curves
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(lstm_history,        label='LSTM',        color='steelblue',  linewidth=2)
ax.plot(transformer_history, label='Transformer', color='darkorange', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss')
ax.set_title('LSTM vs Transformer — Sentiment Classification', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('lstm_vs_transformer.png', dpi=150, bbox_inches='tight'); plt.show()

<a name="skip-code-27"></a>

**Expected output:** Accuracy comparison and learning curves for both models. On a dataset of 80 sentences, the two models likely perform similarly — this is expected, not a failure. What to observe instead: how does the Transformer's learning curve compare? Does it converge differently? Is it more or less stable than the LSTM?

**The real advantage of Transformers emerges at scale** — BERT, GPT, and T5 are all Transformer-based and trained on billions of tokens, a scale of parallelism that would be computationally impossible with LSTMs. The architecture unlocked scale; scale unlocked capability.

<nav aria-label="Return navigation for code cell 27">
<a href="#read-code-27" aria-label="Go back and read code cell 27">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-28"></a>

**Cell 28 — Visualize Multi-Head Attention Weights**

This cell hooks into the Transformer's attention mechanism to extract and visualize the attention weights for a sample sentence. Each head in the multi-head attention learns different attention patterns — visualizing them shows what the model learned to focus on.

<nav aria-label="Code cell 28 navigation">
<a href="#skip-code-28" aria-label="Skip code cell 28 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
transformer_model.eval()

# Hook to capture attention weights
attn_weights_captured = {}

def get_attention_hook(name):
    def hook(module, input, output):
        # TransformerEncoderLayer returns (output,) or (output, attn_weights)
        # We need to call it with need_weights=True separately
        pass
    return hook

# Get attention weights manually
sample_sentence = 'this film was absolutely wonderful and moving'
sample_tensor   = torch.tensor([encode(sample_sentence)], dtype=torch.long)
tokens          = sample_sentence.split()[:MAX_LEN]

with torch.no_grad():
    emb    = transformer_model.embedding(sample_tensor)
    emb_pe = transformer_model.pos_enc(emb)
    
    # Access the first encoder layer's attention directly
    first_layer = transformer_model.transformer_encoder.layers[0]
    attn_out, attn_w = first_layer.self_attn(
        emb_pe, emb_pe, emb_pe, need_weights=True
    )

attn_matrix = attn_w[0].detach().numpy()  # (seq, seq)
n_tok = min(len(tokens), MAX_LEN)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Head 1 (averaged across all heads)
im = axes[0].imshow(attn_matrix[:n_tok, :n_tok], cmap='Blues', vmin=0)
axes[0].set_xticks(range(n_tok)); axes[0].set_xticklabels(tokens[:n_tok], rotation=45, ha='right', fontsize=9)
axes[0].set_yticks(range(n_tok)); axes[0].set_yticklabels(tokens[:n_tok], fontsize=9)
axes[0].set_title('Self-Attention Weights — Layer 1\n(row = query token, col = attended token)', fontsize=10)
plt.colorbar(im, ax=axes[0])

# Attention entropy per token (how focused vs spread out is attention?)
entropy = -np.sum(attn_matrix[:n_tok, :n_tok] * np.log(attn_matrix[:n_tok, :n_tok] + 1e-9), axis=-1)
axes[1].bar(range(n_tok), entropy[:n_tok], color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_xticks(range(n_tok)); axes[1].set_xticklabels(tokens[:n_tok], rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Attention Entropy'); axes[1].set_title('Per-Token Attention Entropy\n(low = focused, high = spread)', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle(f'Sentence: "{sample_sentence}"', fontsize=11, y=1.02)
plt.tight_layout(); plt.savefig('transformer_attention.png', dpi=150, bbox_inches='tight'); plt.show()

<a name="skip-code-28"></a>

**Expected output:** An attention heatmap and entropy bar chart for the sample sentence. The heatmap shows which token each query token attends to most strongly. High entropy (right chart) means the attention is spread across many tokens — the model is uncertain which tokens are most relevant for that position. Low entropy means focused attention — the model has learned to look at one specific token.

**Accessibility note:** Add a Markdown cell describing the attention patterns — which words attend most strongly to each other? Does 'wonderful' attend to 'absolutely'? Does 'film' attend to 'this'?

<nav aria-label="Return navigation for code cell 28">
<a href="#read-code-28" aria-label="Go back and read code cell 28">&#8617; Go back and read the code cell</a>
</nav>

---
# Workshop Preparation Summary

You have now built and trained every major neural network architecture. Here is the conceptual through-line:

## The Architecture Story

| Architecture | The Key Idea | The Problem It Left Open |
| :--- | :--- | :--- |
| **ANN** | Neurons compose into functions. Hidden layers = feature detectors. | No spatial or sequential awareness |
| **CNN** | Shared local filters detect spatial patterns anywhere in the image. | No memory across time |
| **RNN** | Hidden state carries memory through a sequence. | Vanishing gradients over long sequences |
| **LSTM** | Gated cell state preserves long-range dependencies. | Sequential processing limits parallelism |
| **Seq2Seq** | Encoder compresses input; decoder generates output. | Fixed-size bottleneck loses long-sequence context |
| **Attention** | Dynamic weighting over all encoder states per decoding step. | Still depends on recurrent backbone |
| **Transformer** | Self-attention removes recurrence entirely. All positions attend to all positions in parallel. | Quadratic memory cost (solved by efficient attention variants) |

## The Hidden Layer Principles (Your Reference Card)

- **First hidden layer:** 2× to 4× input size
- **Subsequent layers:** halve each time (funnel pattern)
- **Minimum:** never smaller than the output
- **Powers of 2:** 16, 32, 64, 128, 256 — GPU-friendly
- **Activation:** ReLU for hidden layers; let the loss function handle the output
- **BatchNorm:** between linear and activation
- **Dropout:** after activation, before next linear
- **Depth vs Width:** go deeper before wider — depth adds abstraction, width adds capacity

## What to Expect in the Transformer Workshop

The workshop will go deeper on:

- **BERT:** Bidirectional Transformer pre-training — how masked language modeling works
- **GPT:** Causal (autoregressive) Transformer — the decoder-only architecture
- **Fine-tuning:** How to adapt a pre-trained Transformer to a specific task in minutes
- **Efficient Attention:** How researchers address the quadratic memory cost
- **The emergent capabilities debate:** Why do larger Transformers suddenly gain new abilities?

You are now prepared to engage with all of these topics because you built the foundation yourself.

---

## Key Interview Questions You Can Now Answer

- *'Why do we use ReLU instead of sigmoid in hidden layers?'*
  → Sigmoid saturates (derivative ≈ 0 at extremes) — gradients vanish. ReLU has derivative 1 for positive inputs.

- *'What does a hidden layer actually learn?'*
  → A learned linear transformation followed by non-linearity — each neuron is a feature detector. Deeper layers detect increasingly abstract features.

- *'Why does the Transformer outperform LSTMs on long sequences?'*
  → Self-attention creates a direct connection between any two positions in one step. An LSTM's gradient must flow through every intermediate step, which vanishes over distance.

- *'What is multi-head attention and why does it help?'*
  → Running attention multiple times with different projections lets the model simultaneously attend to different types of relationships in the data.

- *'Why do Transformers need positional encoding?'*
  → Self-attention is permutation-invariant — it treats [A,B,C] and [C,B,A] identically without explicit position information injected into the embeddings.

---

**To share:** Go to **Share** in Colab → set access to **Anyone with the link can view** → copy the link → paste into Canvas.